# DermAI V2: Massive Multi-Dataset Training (ISIC 2019, 2020, 2024)

This notebook is designed to train the `DermAIEngine` on a massive combined dataset to push clinical accuracy to the maximum possible limit.

### ⚠️ CRITICAL WARNING: DISK SPACE & COMPUTE ⚠️
Combining ISIC 2019, 2020, and 2024 requires **hundreds of gigabytes** of disk space and massive GPU compute.
- A standard free Google Colab instance will likely run out of disk space (`Disk out of space` error).
- You will need **Colab Pro / Pro+** with a High-RAM, Premium GPU (A100), and expanded disk space, or a dedicated cloud instance (AWS/GCP/RunPod) to process all this data simultaneously.

### Prerequisites:
1. **Kaggle API Key:** You need your `kaggle.json` file.
2. **Accept Competition Rules:** You MUST visit and accept the rules for both competitions to avoid `403 Forbidden` errors:
   - [SIIM-ISIC Melanoma Classification (2020)](https://www.kaggle.com/competitions/siim-isic-melanoma-classification/rules)
   - [ISIC 2024 Challenge](https://www.kaggle.com/competitions/isic-2024-challenge/rules)

In [ ]:
!pip install torch torchvision transformers pandas scikit-learn pillow kaggle tqdm -q

import os
from google.colab import files

print('Please upload your kaggle.json file:')
uploaded = files.upload()

!mkdir -p ~/.kaggle/ && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

print('Downloading Datasets... This will take a LONG time and require massive disk space.')
# 1. ISIC 2019 Dataset
!kaggle datasets download -d salviohexia/isic-2019-skin-lesion-images-for-classification -p data/isic2019
!unzip -q data/isic2019/*.zip -d data/isic2019/extracted

# 2. ISIC 2020 (SIIM) Competition
!kaggle competitions download -c siim-isic-melanoma-classification -p data/isic2020
!unzip -q data/isic2020/*.zip -d data/isic2020/extracted

# 3. ISIC 2024 Competition
!kaggle competitions download -c isic-2024-challenge -p data/isic2024
!unzip -q data/isic2024/*.zip -d data/isic2024/extracted

print('All datasets downloaded and extracted!')

In [ ]:
import pandas as pd
import os

# --- Unified Data Processing ---
# Because different years use different CSV structures, we must unify them into a single format.
unified_records = []

print("Processing ISIC 2020...")
if os.path.exists('data/isic2020/extracted/train.csv'):
    df_2020 = pd.read_csv('data/isic2020/extracted/train.csv')
    for _, row in df_2020.iterrows():
        unified_records.append({
            'image_path': f"data/isic2020/extracted/jpeg/train/{row['image_name']}.jpg",
            'age': row.get('age_approx', 45.0),
            'sex': str(row.get('sex', 'unknown')).lower(),
            'anatomy': str(row.get('anatom_site_general_challenge', 'unknown')).lower(),
            'target': float(row.get('target', 0)) # 1 = malignant, 0 = benign
        })

print("Processing ISIC 2024...")
if os.path.exists('data/isic2024/extracted/train-metadata.csv'):
    df_2024 = pd.read_csv('data/isic2024/extracted/train-metadata.csv')
    for _, row in df_2024.iterrows():
        unified_records.append({
            'image_path': f"data/isic2024/extracted/train-image/image/{row['isic_id']}.jpg",
            'age': row.get('age_approx', 45.0),
            'sex': str(row.get('sex', 'unknown')).lower(),
            'anatomy': str(row.get('anatom_site_general', 'unknown')).lower(),
            'target': float(row.get('target', 0))
        })

unified_df = pd.DataFrame(unified_records)
# Drop rows where the image file doesn't actually exist (sanity check)
unified_df = unified_df[unified_df['image_path'].apply(os.path.exists)]
unified_df['age'] = unified_df['age'].fillna(45.0)

print(f"Total unified training samples available: {len(unified_df)}")
unified_df.to_csv('unified_training_data.csv', index=False)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from PIL import Image
from torchvision import transforms
from transformers import ViTModel, ViTConfig
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# 1. Re-define the DermAI Architecture
class MetadataCrossAttention(nn.Module):
    def __init__(self, embed_dim, metadata_dim):
        super().__init__()
        self.metadata_proj = nn.Linear(metadata_dim, embed_dim)
        self.query_proj = nn.Linear(embed_dim, embed_dim)
        self.key_proj = nn.Linear(embed_dim, embed_dim)
        self.value_proj = nn.Linear(embed_dim, embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, metadata):
        m_proj = self.metadata_proj(metadata).unsqueeze(1)
        q = self.query_proj(m_proj)
        k = self.key_proj(x)
        v = self.value_proj(x)
        attn_out, _ = self.attn(q, k, v)
        return self.norm(m_proj + attn_out).squeeze(1)

class DermAIEngine(nn.Module):
    def __init__(self, metadata_dim=4):
        super().__init__()
        self.config = ViTConfig.from_pretrained('google/vit-base-patch16-224-in21k')
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        self.cross_attn = MetadataCrossAttention(self.config.hidden_size, metadata_dim)
        self.risk_head = nn.Linear(self.config.hidden_size, 1)
        self.mutation_head = nn.Linear(self.config.hidden_size, 1)

    def forward(self, pixel_values, metadata):
        patch_embeddings = self.vit(pixel_values=pixel_values).last_hidden_state
        fused = self.cross_attn(patch_embeddings, metadata)
        return torch.sigmoid(self.risk_head(fused)), torch.sigmoid(self.mutation_head(fused))


In [ ]:
# 2. Build the Unified Dataloader
class UnifiedISICDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): 
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        if self.transform: image = self.transform(image)
        
        # Encode Metadata
        age_norm = float(row['age']) / 100.0
        sex_enc = 1.0 if row['sex'] == 'male' else 0.0
        
        loc_map = {'head/neck': 0.0, 'torso': 0.25, 'upper extremity': 0.5, 'lower extremity': 0.75, 'palms/soles': 1.0, 'acral': 1.0}
        loc_enc = 0.5
        for k, v in loc_map.items():
            if k in row['anatomy']:
                loc_enc = v
                break
                
        md = torch.tensor([age_norm, sex_enc, loc_enc, 0.0], dtype=torch.float32)
        risk = torch.tensor([row['target']], dtype=torch.float32)
        mut = torch.tensor([0.0], dtype=torch.float32) # Standard ISIC lacks BRAF status
        
        return image, md, risk, mut

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_df, val_df = train_test_split(unified_df, test_size=0.1, stratify=unified_df['target'], random_state=42)

train_loader = DataLoader(UnifiedISICDataset(train_df, transform), batch_size=32, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(UnifiedISICDataset(val_df, transform), batch_size=64, num_workers=4, pin_memory=True)


In [ ]:
# 3. High-Performance Training Loop
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = DermAIEngine(metadata_dim=4).to(device)

# To prevent catastrophic forgetting and speed up early training, 
# freeze the heavy ViT backbone initially and train only the cross-attention & heads.
for param in model.vit.parameters(): 
    param.requires_grad = False

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
# Use BCEWithLogitsLoss if we didn't have sigmoid in forward, but we have sigmoid so BCELoss
criterion = nn.BCELoss()

epochs = 5 # Start with 5 epochs for the heads, then unfreeze ViT for fine-tuning
best_val_auc = 0.0

print("Starting Phase 1: Training Custom Cross-Attention & Heads...")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for imgs, mds, risks, _ in loop:
        optimizer.zero_grad()
        risk_pred, _ = model(imgs.to(device), mds.to(device))
        loss = criterion(risk_pred, risks.to(device))
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for imgs, mds, risks, _ in val_loader:
            risk_pred, _ = model(imgs.to(device), mds.to(device))
            preds.extend(risk_pred.cpu().numpy())
            labels.extend(risks.cpu().numpy())
            
    val_auc = roc_auc_score(labels, preds)
    print(f"\n>>> Epoch {epoch+1} Completed | Validation ROC-AUC: {val_auc:.4f} <<<\n")
    
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), 'engine_isic_best.pth')


In [ ]:
# 4. Save and Download the Ultimate Weights
print(f"Training complete! Best Validation AUC: {best_val_auc:.4f}")
print("Downloading the highly accurate weights...")
from google.colab import files
files.download('engine_isic_best.pth')
